# Two-band k.p model -- Table II and Figure 6

The trigonal warping $\kappa$ of the tight-binding models by Löwdin reduction at $K$, the k.p spectra of Fig. 6, and the threshold coefficient of the k.p model with first-principles parameters (the squares of Fig. 11). The reduction and the k.p kernels are in the engine.

## Load engine

In [ ]:
%run engine.ipynb

## Resolution

Paper settings, with a quick setting in the comment. The cached results in `data/` have the resolution in their file names, so changing a line here starts a fresh run for that section.

In [ ]:
N_kp = 401         # k.p square grid across the disk (Figs. 6 and 11); 301 for a quick look (coarser grids ripple)

## Trigonal warping (Table II)

Active spin (the band-edge block), physical $\lambda$. $v$ in eV Å and $\kappa$ in eV Å$^2$.

In [ ]:
kap = {}
for func in ("GGA", "LDA"):
    for model in ("TNN", "NN"):
        for m in materials:
            kap[m, model, func] = warping(m, func, model)

print("GGA      v TNN   kappa TNN   kappa NN   kappa NN (closed form)")
for m in materials:
    r = lowdin(m)
    print(f"{m:6s} {r['v'] * r['a']:7.3f} {kap[m, 'TNN', 'GGA']:+10.4f} {kap[m, 'NN', 'GGA']:+10.4f} {kappa_NN_closed(m):+12.4f}")
print("LDA kappa TNN:", np.round([kap[m, "TNN", "LDA"] for m in materials], 4))
print("LDA kappa NN: ", np.round([kap[m, "NN", "LDA"] for m in materials], 4))

## Figure 6 -- k.p spectra

Parameters from the reduction of the MoS$_2$ TNN model at $\lambda = 0$, with $\lambda$ put back in $d_z$; $k$ in $1/a$, disk of radius $0.6/a$, $\nu = 10$ meV. Single valley at $B = 0$, both valleys at $B = 0$ (the injection current cancels), and both valleys at 30 T. About 10 s at paper settings.

In [ ]:
kp = lowdin("MoS2", lam=0.0)
kp["lam"] = lambda_SOC["MoS2"]
a = kp["a"]
print(f"MoS2 k.p: v = {kp['v'] * a:.3f} eV A, kappa = {kp['kappa'] * a**2:.4f} eV A^2, w = {kp['w'] * a**2:.3f} eV A^2, m_D = {kp['mD']:.4f} eV")

Eg_kp = 2 * kp["mD"] - kp["lam"]
omega_kp = np.linspace(Eg_kp - 0.1, Eg_kp + 1.0, 221)
fn = f"data/kp_spectra_N{N_kp}.npz"
if not os.path.exists(fn):
    out = {}
    runs = [("K_B0", 0.0, [(1, 1)]),
            ("KKp_B0", 0.0, [(1, 1), (-1, -1)]),
            ("KKp_B30", mu_B * 30, [(1, 1), (-1, -1)])]
    for tag, b, pairs in runs:
        eta, sigma = kp_currents(kp, b, omega_kp, 0.01, 0.6, N_kp, pairs, shift=True)
        out["eta_" + tag] = eta_SI(a) * eta[:, 0, 0, 0]
        out["shift_" + tag] = shift_SI(a) * sigma[:, 1, 1, 1]
    np.savez(fn, **out)
spectra = np.load(fn)

In [ ]:
shown = omega_kp <= Eg_kp + 0.45
fig, ax = plt.subplots(1, 2, figsize=(TW, 2.45), gridspec_kw=dict(wspace=0.42))

a1 = ax[0]
a1.plot(omega_kp[shown], spectra["eta_K_B0"][shown], "--", color=BLUE, label=r"$K$, $B=0$ (left)")
a1.plot(omega_kp[shown], spectra["eta_KKp_B0"][shown], color=GREY, label=r"$K+K'$, $B=0$ (left)")
a2 = a1.twinx()
a2.plot(omega_kp[shown], 1e3 * spectra["eta_KKp_B30"][shown], color=RED, label=r"$K+K'$, $B=30$ T (right)")
a1.set_ylabel(r"$\eta^{x;xx}$ ($%s$)" % UNIT)
a2.set_ylabel(r"$\eta^{x;xx}_{K+K'}$ ($10^{-3}\,%s$)" % UNIT, color=RED)
a2.tick_params(axis="y", which="both", colors=RED)
a2.spines["right"].set_color(RED)
a1.tick_params(right=False, which="both")
for axis, y in ((a1, spectra["eta_K_B0"][shown]), (a2, 1e3 * spectra["eta_KKp_B30"][shown])):
    top = np.abs(y).max()
    axis.set_ylim(-0.05 * top, 1.42 * top)
h1, l1 = a1.get_legend_handles_labels()
h2, l2 = a2.get_legend_handles_labels()
a1.legend(h1 + h2, l1 + l2, loc="upper left")

b1 = ax[1]
b1.plot(omega_kp[shown], spectra["shift_K_B0"][shown], "--", color=BLUE, label=r"$K$, $B=0$")
b1.plot(omega_kp[shown], spectra["shift_KKp_B0"][shown], color=GREY, lw=2.2, label=r"$K+K'$, $B=0$")
b1.plot(omega_kp[shown], spectra["shift_KKp_B30"][shown], color=RED, lw=0.9, label=r"$K+K'$, $B=30$ T")
b1.set_ylabel(r"$\sigma^{y;yy}$ ($%s$)" % SUNIT)
b1.legend(loc="lower right")
b1.set_ylim(bottom=0)

for i, axis in enumerate(ax):
    zero(axis)
    axis.set_xlabel(r"$\hbar\omega$ (eV)")
    label(axis, "ab"[i])
save(fig, "kp_spectra")
plt.show()

ratio = np.abs(spectra["eta_KKp_B30"][shown]).max() / np.abs(spectra["eta_K_B0"][shown]).max()
print(f"The K+K' peak at 30 T is {100 * ratio:.2f}% of the single-valley peak.")

## Threshold $\chi$ with first-principles parameters

The squares of Fig. 11: $\chi = \eta^{x;xx}(30\,{\rm T})/30$ T at $E_A + 0.01$ eV. `kp_chi` saves the result in `data/` for `edge.ipynb`. A few seconds.

In [ ]:
chi_kp = kp_chi(N_kp)
for m in materials:
    p = kp_first_principles(m)
    print(f"{m}: w = {p['w']:.3f} eV A^2, m_D = {p['mD']:.4f} eV, chi = {1e4 * chi_kp[m]:.3f} x 1e-4 A m V^-2 s^-1 T^-1")